# Gluon PDF from the double ratio: one-loop matching and fit

Starts from the double ratio (section I1 of `matrix_element_analysis_final.ipynb`: the files `itd_{method}_...h5`) and ends with the gluon PDF. Forward frame, pion or proton.

| section | what it does |
|---|---|
| 0 | settings: hadron, which double ratio (HYP 8, one flow time, or extrapolated to zero flow time), folders; the functions |
| P1 | the **lattice data**: read the double ratio of the methods, go to zero flow time, average the methods (if there are several), choose the points |
| P2 | the **matching**: the matrix $K$ (one number per point and $x$ node), checked against the direct integrals |
| P3 | the **fit** in $\nu$ space, every jackknife sample: $y^{\rm mod}$ (pure theory with correction terms) against the lattice data |
| P4 | check of the fit: lattice data, pure theory and model at every point |
| P5 | the **PDF** $f(x) = x g(x) / \langle x \rangle_g$ in $x$ space, statistical and systematic error, plot |
| P6 | plot of several saved PDFs together |

The formulas and the names ($y^{\rm lat}$, $y^{\rm th}$, $y^{\rm mod}$, $K$, $W_k$) are those of `~/Desktop/explaination_docs/gluon_pdf_fit_theory_and_lattice.pdf`:

$$ f(x) = \frac{x^{\alpha}(1-x)^{\beta}}{B(\alpha+1,\beta+1)}, \qquad I(\nu) = \int_0^1 dx\, \cos(x\nu) f(x) $$

$$ y^{\rm th}(\nu, z) = I(\nu) - \frac{\alpha_s N_c}{2\pi} \int_0^1 du\, \left[ L(z) B(u) + V(u) + P(u) \right] \left[ I(u\nu) - I(\nu) \right] $$

$$ y^{\rm mod} = 1 - (1 - y^{\rm th}) \left[ 1 + c_L L(z) + c_{z2} w^2 + c_a (a p_z)^2 \right] $$

The methods are chosen in section 0 (`methods`); the default is the sGEVP alone. The correction terms that are used are chosen in P3 (`fit_terms`); the default is $c_{z2}$ and $c_a$, without the log term $c_L$.

**How to run it.** Section 0 first, then top to bottom. P1 saves the lattice data, P3 the fits, P5 the PDF; after a restart run section 0, P1 and P2, then the section you want. The fits take about 30 s.

**Shapes.** Every array has its shape in a comment, each axis as name:size with the sizes of the default run.

## 0. Settings: hadron, which double ratio, folders; the functions used by several sections

In [ ]:
#======== 0. settings, and the functions used by several sections ========
import os
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"      #one BLAS thread per process: the parallel fits run faster;  works only before numpy is imported (a fresh kernel)
os.environ["OMP_NUM_THREADS"] = "1"
import time
import h5py
import numpy as np
import matplotlib.pyplot as plt
import gvar as gv
import lsqfit
from joblib import Parallel, delayed            #runs the fits in n_workers processes at once
from scipy.integrate import quad                #only for the check of P2
from scipy.special import beta as euler_beta    #B(a, b), the Euler beta function

#---- what to analyse
hadron = "pion"                     #"pion" or "proton" (the proton files are projected to positive parity)
smearing = "flow"                   #"flow": the gluon operator at Wilson-flow times;  "hyp8": the HYP 8 double ratio of matrix_element_analysis_final.ipynb
flow_times = [2.0, 2.5, 3.0, 3.5, 4.0]          #t/a^2 of the levels that are used (smearing = "flow")
flow_degree = 1                     #degree of the polynomial in t/a^2 through these levels (least squares, every level the same weight);  its value at t = 0 is used
#  one flow time without extrapolation:  flow_times = [3.0], flow_degree = 0
#  zero flow time, line:                 flow_times = [2.0, 2.5, 3.0, 3.5, 4.0], flow_degree = 1
#  zero flow time, quadratic:            flow_times = [2.0, 2.5, 3.0, 3.5, 4.0], flow_degree = 2

#---- the methods of the matrix elements
methods = ["sgevp"]                 #one method: used alone (statistical error only);  several, e.g. ["ratio_separate", "ratio_shared", "sgevp"]: their average, and their spread is the systematic error
sgevp_only_pz = {"pion": [], "proton": [6]}     #several methods: at these pz their average is the sGEVP alone
sgevp_only_in_method_fits = False   #True: at these pz the fits to one method alone (systematic error, P5) use the sGEVP too;  False: they use their own points

#---- the lattice
Ls = 32
a_fm = 0.09                         #lattice spacing in fm
hbarc = 0.1973269804                #GeV fm

#---- running
n_workers = 4                       #fits that run at the same time (one per data set);  1 = one after the other, in this kernel
show_plots = True                   #True = every figure is also drawn here;  False = only saved

#---- folders and file names
analysis_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_analysis"     #fits, results, plots of this notebook;  the HYP 8 double ratio
flow_dir = "/Users/mcp3270/Desktop/flow_ncfg1504"                                               #the double ratio at the flow times
ncfg = 1504
hadron_tag = {"pion": "pion", "proton": "proton_projected"}[hadron]
run_tag = f"forward_ncfg{ncfg}_{hadron_tag}"                             #the end of every file name
save_dir = os.path.join(analysis_dir, f"{hadron}_forward_ncfg{ncfg}")
hyp8_itd_name = os.path.join(save_dir, "results", "itd_{method}_" + run_tag + ".h5")
flow_itd_name = os.path.join(flow_dir, f"{hadron}_forward_ncfg{ncfg}", "flow_t{t}", "results", "itd_{method}_" + run_tag + ".h5")

#---- data_tag: the name of the chosen double ratio in every file name and plot title
if smearing == "hyp8":
    data_tag = "hyp8"
elif len(flow_times) == 1:
    data_tag = f"flow_t{flow_times[0]}"
else:
    data_tag = "zero_flow_" + {1: "line", 2: "quadratic"}[flow_degree] + f"_t{flow_times[0]}-{flow_times[-1]}"
if len(methods) == 1:
    main_data_set = methods[0]      #the data set of the result:  the method itself ...
    data_sets = methods             #the data sets that are fitted in P3
else:
    main_data_set = "average"       #... or the average of the methods
    data_sets = ["average"] + methods
lattice_name = "pdf_matching/lattice_data_" + data_tag + "_" + main_data_set + "_" + run_tag + ".h5"                        #P1
fit_name = "pdf_matching/fits/pdf_fit_" + data_tag + "_{data_set}_w{w0}-{w1}_pz{p0}-{p1}_{terms}_svd{svdcut:g}_" + run_tag + ".npz"    #P3
pdf_name = "pdf_matching/gluon_pdf_" + data_tag + "_" + main_data_set + "_w{w0}-{w1}_pz{p0}-{p1}_{terms}_svd{svdcut:g}_" + run_tag + ".h5"         #P5
for folder in ["pdf_matching", "pdf_matching/fits", "pdf_matching/plots"]:
    os.makedirs(os.path.join(save_dir, folder), exist_ok=True)

assert hadron in ["pion", "proton"], f"hadron = {hadron}: use pion or proton"
assert smearing in ["flow", "hyp8"], f"smearing = {smearing}: use flow or hyp8"
assert smearing == "hyp8" or len(flow_times) > flow_degree, f"{len(flow_times)} flow times are too few for a polynomial of degree {flow_degree}"
assert smearing == "hyp8" or len(flow_times) == 1 or flow_degree in [1, 2], "several flow times: flow_degree must be 1 (line) or 2 (quadratic)"


#======== functions used by several sections ========
def jk(x):
    #jackknife mean and error over the first axis:  error = sqrt((N - 1)/N sum_j (x_j - mean)^2)
    n = x.shape[0]
    mean = x.mean(axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((x - mean) ** 2, axis=0))
    return mean, err


def excess_spread(x):
    #systematic error from the methods:  x (method:3, jk:1504, ...);  how much more the method means differ than their statistical errors allow
    n_method, n_jk = x.shape[:2]
    method_mean = x.mean(axis=1)                                        # (method:3, ...)
    s2 = method_mean.var(axis=0, ddof=1)                                # (...)  sample variance of the method means
    d = x - method_mean[:, None]                                        # (method:3, jk:1504, ...)
    C = (n_jk - 1) / n_jk * np.einsum("mj...,nj...->...mn", d, d)       # (..., method:3, method:3)  jackknife covariance of the methods
    noise = (np.trace(C, axis1=-2, axis2=-1) - C.sum(axis=(-2, -1)) / n_method) / (n_method - 1)     # (...)  what s2 would be from statistics alone
    return np.sqrt(np.maximum(0.0, s2 - noise))                         # (...)


def fit_jackknife(y_jk, x, model, prior, prior_args, starts, names, svdcut, maxit, tol, fitter):
    #the fit_jackknife of matrix_element_analysis_final.ipynb (section 0), without the neglog parameters:  one fit in every jackknife sample
    #  y_jk (jk:1504, point:n): the data of every sample;  x: the x of every point (a dictionary of arrays)
    #  model(x, p): the fit function;  prior(*prior_args): the prior (made again after every gv.switch_gvar)
    #  starts: the start values of the fit of the mean (a list of dictionaries);  names: the parameters that are kept
    #the fit of the mean uses the covariance of all cfgs and keeps the lowest chi2 of the starts;
    #jackknife sample j starts from it and uses the covariance of the other cfgs (rebuilt without cfg j)
    n_jk, n_point = y_jk.shape
    #---- the fit of the mean
    y_mean = y_jk.mean(axis=0)                                          # (point:n)
    d = y_jk - y_mean                                                   # (jk:1504, point:n)
    cov_all = (n_jk - 1) / n_jk * (d.T @ d)                             # (point:n, point:n)  jackknife covariance of the mean
    best = None
    chi2_starts = []
    for p0 in starts:
        gv.switch_gvar()
        fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_mean, cov_all)), fcn=model, prior=prior(*prior_args), p0=p0,
                                   svdcut=svdcut, maxit=maxit, tol=tol, fitter=fitter)
        chi2_starts.append(float(fit.chi2))
        if best is None or fit.chi2 < best["chi2"]:
            best = {"chi2": float(fit.chi2), "dof": int(fit.dof), "svdn": int(fit.svdn), "at_maxit": int(fit.stopping_criterion == 0),
                    "p0": {key: np.array(fit.pmean[key]) for key in fit.pmean}}
        gv.restore_gvar()
    #---- every jackknife sample:  cov_j = covariance of the per-cfg values without cfg j, divided by the number of cfgs - 1
    y_cfg = n_jk * y_mean - (n_jk - 1) * y_jk                           # (cfg:1504, point:n)  the per-cfg values, rebuilt from the samples
    y_c = y_cfg - y_cfg.mean(axis=0)                                    # (cfg:1504, point:n)
    Q = y_c.T @ y_c                                                     # (point:n, point:n)
    m = n_jk - 1
    param_jk = {}
    for name in names:
        param_jk[name] = []
    chi2_jk = np.zeros(n_jk)                                            # (jk:1504)
    n_at_maxit = best["at_maxit"]
    for j in range(n_jk):
        cov_j = (Q - (1 + 1 / m) * np.outer(y_c[j], y_c[j])) / (m - 1) / (n_jk - 1)     # (point:n, point:n)
        gv.switch_gvar()
        fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_jk[j], cov_j)), fcn=model, prior=prior(*prior_args), p0=best["p0"],
                                   svdcut=svdcut, maxit=maxit, tol=tol, fitter=fitter)
        for name in names:
            param_jk[name].append(np.array(fit.pmean[name]))
        chi2_jk[j] = fit.chi2
        n_at_maxit = n_at_maxit + int(fit.stopping_criterion == 0)
        gv.restore_gvar()
    for name in names:
        param_jk[name] = np.array(param_jk[name])                       # (jk:1504)
    #---- the fitted curve of every sample and the pulls:  (data mean - curve) / jackknife error of the point
    curve_jk = np.zeros((n_jk, n_point))                                # (jk:1504, point:n)
    for j in range(n_jk):
        p_j = {}
        for name in names:
            p_j[name] = param_jk[name][j]
        curve_jk[j] = model(x, p_j)
    y_m, y_e = jk(y_jk)                                                 # (point:n), (point:n)
    curve_mean = curve_jk.mean(axis=0)                                  # (point:n)
    return {"param_jk": param_jk, "chi2_jk": chi2_jk, "chi2": best["chi2"], "dof": best["dof"], "svdn": best["svdn"],
            "n_at_maxit": n_at_maxit, "chi2_starts": np.array(chi2_starts), "y_mean": y_m, "y_err": y_e,
            "curve_jk": curve_jk, "curve_mean": curve_mean, "pull": (y_m - curve_mean) / y_e}


def save_figure(fig, name):
    #a figure:  saved as PNG in pdf_matching/plots, then drawn here (show_plots) or closed
    fig.savefig(os.path.join(save_dir, "pdf_matching/plots", f"{name}_{run_tag}.png"), dpi=130)
    if show_plots:
        plt.show()
    else:
        plt.close(fig)


print(f"{hadron_tag} | double ratio: {data_tag} | methods {methods}, result: {main_data_set}")
print("folder", os.path.join(save_dir, "pdf_matching"))

## P1. Lattice data: the double ratio of the methods, zero flow time, the average of the methods

In [ ]:
#======== P1. lattice data:  read the double ratio of every method (I1 files), go to zero flow time, average the methods;  saved ========
#double ratio(pz, w) = [M(pz, w) / M(pz, 0)] / [M(0, w) / M(0, 0)] in every jackknife sample;  no PDF and no matching in it

#---- the levels that are read, and the weight c of each level:  the polynomial fit through the levels, its value at t = 0, is a sum  sum_l c_l y_l
if smearing == "hyp8":
    levels = ["hyp8"]
    level_weight = np.array([1.0])                                          # (level:1)
else:
    levels = flow_times
    X = np.vander(np.array(flow_times), flow_degree + 1, increasing=True)   # (level:5, power:3)  1, t, t^2
    level_weight = np.linalg.pinv(X)[0]                                     # (level:5)  least squares:  the coefficient of t^0
print("weight of every level at zero flow time:", dict(zip(levels, np.round(level_weight, 4))))

#---- read every method and level
double_ratio = {}               #double_ratio[method]: (jk:1504, w:10, pz:7)
cfg_list = None
for method in methods:
    level_data = []
    for level in levels:
        if smearing == "hyp8":
            file = hyp8_itd_name.format(method=method)
        else:
            file = flow_itd_name.format(method=method, t=level)
        with h5py.File(file, "r") as f:
            level_data.append(f["double_ratio_jk"][()])                     # (jk:1504, w:10, pz:7)
            nu = f["nu"][()]                                                # (w:10, pz:7)
            pz_list = f["pz_list"][()]                                      # (pz:7)
            cfg_file = f["cfg_list"][()]                                    # (cfg:1504)
        if cfg_list is None:
            cfg_list = cfg_file
        assert np.array_equal(cfg_file, cfg_list), f"{file} has other cfgs"
    level_data = np.array(level_data)                                       # (level:5, jk:1504, w:10, pz:7)
    double_ratio[method] = np.einsum("l,ljwp->jwp", level_weight, level_data)     # (jk:1504, w:10, pz:7)

#---- several methods:  their average in every jackknife sample
if len(methods) > 1:
    x_methods = np.array([double_ratio[method] for method in methods])      # (method:3, jk:1504, w:10, pz:7)
    double_ratio["average"] = x_methods.mean(axis=0)                        # (jk:1504, w:10, pz:7)
    for pz in sgevp_only_pz[hadron]:
        ipz = list(pz_list).index(pz)
        double_ratio["average"][:, :, ipz] = double_ratio["sgevp"][:, :, ipz]
        if sgevp_only_in_method_fits:
            for method in methods:
                double_ratio[method][:, :, ipz] = double_ratio["sgevp"][:, :, ipz]

#---- save
file = os.path.join(save_dir, lattice_name)
with h5py.File(file, "w") as f:
    for data_set in data_sets:
        f[f"double_ratio_jk/{data_set}"] = double_ratio[data_set]           # (jk:1504, w:10, pz:7)
    f["nu"] = nu                                                            # (w:10, pz:7)
    f["pz_list"] = pz_list
    f["cfg_list"] = cfg_list
    f["level_weight"] = level_weight
    f["levels"] = str(levels)
    f["sgevp_only_pz"] = str(sgevp_only_pz[hadron]) + (" in the average and in every method" if sgevp_only_in_method_fits else " in the average")
print("saved", file)

#---- print:  mean(statistical error) of the data set of the result
mean, err = jk(double_ratio[main_data_set])                                 # (w:10, pz:7), (w:10, pz:7)
n_w = mean.shape[0]
print(f"\n{hadron}, {data_tag}, {main_data_set}:  double ratio(pz, w), rows pz 1-6, columns w 1-{n_w - 1}")
for ipz in range(1, len(pz_list)):
    print(f"   pz {pz_list[ipz]}: " + " ".join(f"{str(gv.gvar(mean[w, ipz], err[w, ipz])):>11s}" for w in range(1, n_w)))

## P2. Matching: the nodes, the integrals $G_B, G_V, G_P$ and the matrix $K$

In [ ]:
#======== P2. matching:  the matrix K_ik = K(x_k nu_i, z_i) of every point i and x node k (no lattice data, no alpha, no beta in it) ========
#Balitsky, Morris, Radyushkin, arXiv:1910.13963, eq. (7.17), gluon-gluon part:
#  y_th(nu, z) = I(nu) - kappa int_0^1 du [L(z) B(u) + V(u) + P(u)] [I(u nu) - I(nu)],   kappa = alpha_s N_c / (2 pi)
#  = int_0^1 dx f(x) K(x nu, z),   K(y, z) = cos y - kappa [L(z) G_B(y) + G_V(y) + G_P(y)],   G_g(y) = int_0^1 du g(u) [cos(u y) - cos y]
alpha_s = 0.3
N_c = 3
mu = 2.0                            #GeV
gamma_E = 0.5772156649015329
n_u_nodes = 200                     #Gauss-Legendre points of the u integral, u = 1 - s^2
n_x_nodes = 96                      #Gauss-Legendre points of the x integral, x = t^4
w_fit = [2, 3, 4, 5, 6]             #the points that are fitted:  these w ...
pz_fit = [1, 2, 3, 4, 5, 6]         #... at these pz

kappa = alpha_s * N_c / (2 * np.pi)

#---- the points:  all w of the first pz, then all w of the next pz, ...
pz_point = []
w_point = []
for pz in pz_fit:
    for w in w_fit:
        pz_point.append(float(pz))
        w_point.append(float(w))
pz_point = np.array(pz_point)                                               # (point:30)
w_point = np.array(w_point)                                                 # (point:30)
ap_point = 2 * np.pi * pz_point / Ls                                        # (point:30)  a pz
nu_point = ap_point * w_point                                               # (point:30)
z_gev = w_point * a_fm / hbarc                                              # (point:30)  z in GeV^-1
L_point = np.log(z_gev ** 2 * mu ** 2 * np.exp(2 * gamma_E) / 4)            # (point:30)

#---- the nodes and weights of the u integral:  int_0^1 du g(u) = int_0^1 ds 2 s g(1 - s^2)  ~  sum_m 2 s_m omega_m g(u_m)
s_node, omega_s = np.polynomial.legendre.leggauss(n_u_nodes)                # (m:200), (m:200)  on [-1, 1]
s_node = 0.5 * (s_node + 1)                                                 # (m:200)  on [0, 1]
omega_s = 0.5 * omega_s                                                     # (m:200)
u_node = 1 - s_node ** 2                                                    # (m:200)
u_weight = 2 * s_node * omega_s                                             # (m:200)

#---- the nodes and weights of the x integral:  int_0^1 dx h(x) = int_0^1 dt 4 t^3 h(t^4)  ~  sum_k J_k h(x_k)
t_node, omega_t = np.polynomial.legendre.leggauss(n_x_nodes)                # (k:96), (k:96)  on [-1, 1]
t_node = 0.5 * (t_node + 1)                                                 # (k:96)  on [0, 1]
omega_t = 0.5 * omega_t                                                     # (k:96)
x_node = t_node ** 4                                                        # (k:96)
J = 4 * t_node ** 3 * omega_t                                               # (k:96)
log_x = np.log(x_node)                                                      # (k:96)
log_1mx = np.log1p(-x_node)                                                 # (k:96)  log(1 - x)


def B_kernel(u):
    return 2 * (1 - u * (1 - u)) ** 2 / (1 - u)


def V_kernel(u):
    return 4 * (u + np.log(1 - u)) / (1 - u)


def P_kernel(u):
    return (2.0 / 3.0) * (1 - u ** 3)


def G_integral(g, y):
    #G_g(y) = int_0^1 du g(u) [cos(u y) - cos y]  for every entry of y (point:30, k:96)
    bracket = np.cos(y[..., None] * u_node) - np.cos(y)[..., None]          # (point:30, k:96, m:200)
    return (bracket * (g(u_node) * u_weight)).sum(axis=-1)                  # (point:30, k:96)


y_table = np.outer(nu_point, x_node)                                        # (point:30, k:96)  y = x nu
G_B = G_integral(B_kernel, y_table)                                         # (point:30, k:96)
G_V = G_integral(V_kernel, y_table)                                         # (point:30, k:96)
G_P = G_integral(P_kernel, y_table)                                         # (point:30, k:96)
K_table = np.cos(y_table) - kappa * (L_point[:, None] * G_B + G_V + G_P)    # (point:30, k:96)


def pure_theory(alpha, beta):
    #y_th of every point for one alpha, beta:  the average of row i of K with the weights W_k
    W = J * np.exp(alpha * log_x + beta * log_1mx)                          # (k:96)
    return (K_table @ W) / W.sum()                                          # (point:30)


#---- check:  the matching formula done directly as two nested integrals, at a few points, for one alpha, beta
check_alpha, check_beta = -0.08, 2.65
check_points = [0, len(w_point) // 2, len(w_point) - 1]


def check_I(nu_value):
    #I(nu) = int dx cos(x nu) f(x), with x = t^4
    def integrand(t):
        x = t ** 4
        return 4 * t ** 3 * np.cos(x * nu_value) * x ** check_alpha * (1 - x) ** check_beta / euler_beta(check_alpha + 1, check_beta + 1)
    return quad(integrand, 0, 1, limit=400, epsabs=1e-12, epsrel=1e-12)[0]


theory = pure_theory(check_alpha, check_beta)                               # (point:30)
print("check of the matching (alpha %.2f, beta %.2f):" % (check_alpha, check_beta))
for i in check_points:
    I_nu = check_I(nu_point[i])
    def integrand(s):
        u = 1 - s ** 2
        return 2 * s * (L_point[i] * B_kernel(u) + V_kernel(u) + P_kernel(u)) * (check_I(u * nu_point[i]) - I_nu)
    direct = I_nu - kappa * quad(integrand, 1e-9, 1, limit=200, epsabs=1e-8, epsrel=1e-8)[0]
    print(f"   pz {pz_point[i]:.0f} w {w_point[i]:.0f} (nu {nu_point[i]:.2f}):  K table {theory[i]:.7f}   nested integrals {direct:.7f}   difference {theory[i] - direct:+.1e}")
    assert abs(theory[i] - direct) < 1e-6, "the K table and the nested integrals differ"

## P3. The fit: every jackknife sample of every data set

In [ ]:
#======== P3. fit of the model to the lattice data in nu space:  every data set (average, and every method alone), every jackknife sample;  one file per data set ========
#  model:  y_mod = 1 - (1 - y_th) [1 + sum over fit_terms],   y_th = pure theory of (alpha, beta)
#  terms:  "c_L": c_L L(z)      a ln z^2 dependence other than that of the matching
#          "c_z2": c_z2 w^2     z^2 dependence (higher twist)
#          "c_a": c_a (a pz)^2  lattice artifact of the momentum
fit_terms = ["c_z2", "c_a"]                     #add "c_L" for the log term;  [] = pure theory only
fit_svdcut = 1e-2                               #eigenvalues of the correlation matrix below svdcut x the largest are raised to it
fit_prior = {"log(alpha+1)": (0.0, 1.0),        #center, width;  alpha + 1 = exp(0 +- 1) > 0
             "log(beta)": (np.log(3.0), 1.0),   #beta = 3 exp(+-1) > 0
             "c_z2": (0.0, 1.0),
             "c_a": (0.0, 1.0),
             "c_L": (0.0, 1.0)}
fit_starts = [(1.0, 3.0), (0.3, 1.0), (2.0, 6.0)]     #(alpha + 1, beta) of every start of the fit of the mean;  the coefficients start at 0;  the lowest chi2 is kept
fit_maxit = 20000
fit_tol = 1e-8
fit_fitter = "scipy_least_squares"

#---- the known numbers of every point
x_fit = {"K": K_table, "c_z2": w_point ** 2, "c_a": ap_point ** 2, "c_L": L_point}     #K (point:30, k:96), the others (point:30)


def double_ratio_model(x, p):
    #y_mod of every point;  works with numbers and with gvars
    W = J * gv.exp((p["alpha+1"] - 1) * log_x + p["beta"] * log_1mx)        # (k:96)
    theory = x["K"] @ W / W.sum()                                           # (point:30)  y_th
    factor = 1.0
    for term in fit_terms:
        factor = factor + p[term] * x[term]                                 # (point:30)
    return 1 - (1 - theory) * factor                                        # (point:30)  y_mod


def double_ratio_prior():
    p = gv.BufferDict()
    for key in ["log(alpha+1)", "log(beta)"] + fit_terms:
        p[key] = gv.gvar(*fit_prior[key])
    return p


#---- the settings, written into every fit file
terms_tag = "_".join(fit_terms) if len(fit_terms) > 0 else "theory_only"
name_args = {"w0": w_fit[0], "w1": w_fit[-1], "p0": pz_fit[0], "p1": pz_fit[-1], "terms": terms_tag, "svdcut": fit_svdcut}
settings_text = (f"{hadron} {data_tag} | w {w_fit} pz {pz_fit} | terms {fit_terms} | svdcut {fit_svdcut:g} | prior " +
                 ", ".join(f"{key} {fit_prior[key][0]:.4f}({fit_prior[key][1]:g})" for key in ["log(alpha+1)", "log(beta)"] + fit_terms) +
                 f" | starts {fit_starts} | maxit {fit_maxit} tol {fit_tol:g} {fit_fitter} | alpha_s {alpha_s:g} mu {mu:g} GeV a {a_fm:g} fm"
                 f" | nodes u {n_u_nodes} x {n_x_nodes} | sGEVP alone at pz {sgevp_only_pz[hadron]}" + (" also in the method fits" if sgevp_only_in_method_fits else ""))


def pdf_fit_one(data_set):
    #the fit of one data set in every jackknife sample;  saves its file and returns one line of text
    time_fit = time.time()
    with h5py.File(os.path.join(save_dir, lattice_name), "r") as f:
        data = f[f"double_ratio_jk/{data_set}"][()]                         # (jk:1504, w:10, pz:7)
    y_jk = np.zeros((data.shape[0], len(w_point)))                          # (jk:1504, point:30)
    for i in range(len(w_point)):
        y_jk[:, i] = data[:, int(w_point[i]), list(pz_list).index(int(pz_point[i]))]
    starts = []
    for a1, b in fit_starts:
        p0 = {"log(alpha+1)": np.log(a1), "log(beta)": np.log(b)}
        for term in fit_terms:
            p0[term] = 0.0
        starts.append(p0)
    result = fit_jackknife(y_jk, x_fit, double_ratio_model, double_ratio_prior, (), starts, ["alpha+1", "beta"] + fit_terms,
                           fit_svdcut, fit_maxit, fit_tol, fit_fitter)
    #---- the pure theory part of every sample
    theory_jk = np.zeros(y_jk.shape)                                        # (jk:1504, point:30)
    for j in range(len(y_jk)):
        theory_jk[j] = pure_theory(result["param_jk"]["alpha+1"][j] - 1, result["param_jk"]["beta"][j])
    #---- save
    file = os.path.join(save_dir, fit_name.format(data_set=data_set, **name_args))
    np.savez(file, **{name + "_jk": result["param_jk"][name] for name in result["param_jk"]}, y_jk=y_jk, model_jk=result["curve_jk"], theory_jk=theory_jk,
             chi2_jk=result["chi2_jk"], chi2=result["chi2"], dof=result["dof"], svdn=result["svdn"], n_at_maxit=result["n_at_maxit"],
             chi2_starts=result["chi2_starts"], pull=result["pull"], w=w_point, pz=pz_point, nu=nu_point, settings=settings_text)
    return f"{data_set}: chi2/dof {result['chi2'] / result['dof']:.3f} (starts {np.round(result['chi2_starts'], 4)}), {result['n_at_maxit']} fits at maxit, {time.time() - time_fit:.0f} s"


#---- the fits that are not saved yet
tasks = []
for data_set in data_sets:
    file = os.path.join(save_dir, fit_name.format(data_set=data_set, **name_args))
    if os.path.exists(file):
        saved = str(np.load(file)["settings"])
        assert saved == settings_text, f"{file} was fitted with other settings:\n  {saved}\ndelete the file to fit it again with the settings of this cell"
    else:
        tasks.append(data_set)
print(f"{len(data_sets)} fits in all, {len(data_sets) - len(tasks)} already saved, {len(tasks)} to do with {n_workers} workers", flush=True)
for line in Parallel(n_jobs=n_workers)(delayed(pdf_fit_one)(task) for task in tasks):
    print("   " + line)

#---- a table of every fit
print(f"\n{hadron}, {data_tag}, w {w_fit[0]}-{w_fit[-1]}, pz {pz_fit[0]}-{pz_fit[-1]}:   data set, chi2/dof, parameters (jackknife mean and error)")
for data_set in data_sets:
    d = np.load(os.path.join(save_dir, fit_name.format(data_set=data_set, **name_args)))
    line = f"   {data_set:15s} {float(d['chi2']) / int(d['dof']):.2f}   at maxit {int(d['n_at_maxit'])}  "
    for name in ["alpha+1", "beta"] + fit_terms:
        mean, err = jk(d[name + "_jk"])
        line = line + f" {name} {gv.gvar(mean, err)} "
    print(line)

## P4. Check the fit: lattice data, pure theory and model at every point

In [ ]:
#======== P4. check of the fit:  the model must go through the lattice data;  one panel per pz, one figure per data set ========
#points: lattice data y_lat with jackknife errors;  solid line and band: the model y_mod (jackknife mean and error);  dashed line: the pure theory y_th at the fitted alpha, beta
check_data_sets = data_sets                     #a figure for each of these;  ["average"] = only the average

print(f"{hadron}, {data_tag}:  data set, points within one error, points above the model, mean of (lattice - model)/error, the same per pz")
for data_set in check_data_sets:
    d = np.load(os.path.join(save_dir, fit_name.format(data_set=data_set, **name_args)))
    lat_mean, lat_err = jk(d["y_jk"])                                       # (point:30), (point:30)
    mod_mean, mod_err = jk(d["model_jk"])                                   # (point:30), (point:30)
    th_mean, th_err = jk(d["theory_jk"])                                    # (point:30), (point:30)
    pull = (lat_mean - mod_mean) / lat_err                                  # (point:30)
    print(f"   {data_set:15s} {int((np.abs(pull) <= 1).sum())}/{len(pull)}   above {int((pull > 0).sum())}   {pull.mean():+.2f}   " +
          " ".join(f"pz {pz} {pull[pz_point == pz].mean():+.2f}" for pz in pz_fit))
    fig, axes = plt.subplots(2, 3, figsize=(11, 6.6), sharex=True, sharey=True, squeeze=False)
    for i_pz in range(len(pz_fit)):
        ax = axes.flat[i_pz]
        s = pz_point == pz_fit[i_pz]                                        # (point:30)  the points of this pz
        ax.fill_between(w_point[s], (mod_mean - mod_err)[s], (mod_mean + mod_err)[s], color="#3b2f63", alpha=0.2, lw=0)
        ax.plot(w_point[s], mod_mean[s], color="#3b2f63", lw=1.6, label="model (theory with correction terms)")
        ax.plot(w_point[s], th_mean[s], color="#1f5fa8", lw=1.2, ls="--", marker="s", ms=4, mfc="white", label=r"pure theory at the fitted $\alpha$, $\beta$")
        ax.errorbar(w_point[s], lat_mean[s], lat_err[s], color="#c8500a", marker="o", ms=4, ls="none", elinewidth=1.0, capsize=1.5, label="lattice data")
        ax.set_title(f"pz = {pz_fit[i_pz]},  mean pull {pull[s].mean():+.2f}", fontsize=9)
        ax.set_xticks(w_fit)
        ax.grid(color="#e1e0d9", lw=0.5)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(labelsize=8)
    for ax in axes[1]:
        ax.set_xlabel("w", fontsize=9)
    for ax in axes[:, 0]:
        ax.set_ylabel("double ratio", fontsize=9)
    axes[0, 0].legend(fontsize=7.5, frameon=False, loc="lower left")
    fig.suptitle(f"{hadron}, {data_tag}, {data_set}:  chi2/dof {float(d['chi2']) / int(d['dof']):.2f},  {int((np.abs(pull) <= 1).sum())} of {len(pull)} points within one error,  "
                 f"mean pull {pull.mean():+.2f}", fontsize=10)
    fig.tight_layout()
    save_figure(fig, f"P4_fit_check_{data_tag}_{data_set}_{terms_tag}_svd{fit_svdcut:g}")

## P5. The PDF in $x$ space: $f(x) = x g(x) / \langle x \rangle_g$ ($\overline{\rm MS}$, 2 GeV)

In [ ]:
#======== P5. the PDF:  f(x) of every jackknife sample from its alpha, beta;  central value and statistical error from the fit of main_data_set;
#         several methods: systematic error = excess spread of the fits to the methods alone (excess_spread, section 0) at every x;  saved ========
x_grid = np.linspace(0.005, 0.995, 199)         # (x:199)
x_print = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
pdf_ylim = (0, 5)


def f_of_x(x, alpha1_jk, beta_jk):
    #f(x) of every sample (jk:1504, x:n);  x (x:n), alpha1_jk = alpha + 1 (jk:1504), beta_jk (jk:1504)
    f_jk = np.zeros((len(alpha1_jk), len(x)))                               # (jk:1504, x:n)
    for j in range(len(alpha1_jk)):
        f_jk[j] = x ** (alpha1_jk[j] - 1) * (1 - x) ** beta_jk[j] / euler_beta(alpha1_jk[j], beta_jk[j] + 1)
    return f_jk


f_jk = {}                       #f_jk[data set]: (jk:1504, x:199)
f_print_jk = {}                 #the same at the x of x_print: (jk:1504, x:9)
for data_set in data_sets:
    d = np.load(os.path.join(save_dir, fit_name.format(data_set=data_set, **name_args)))
    f_jk[data_set] = f_of_x(x_grid, d["alpha+1_jk"], d["beta_jk"])
    f_print_jk[data_set] = f_of_x(np.array(x_print), d["alpha+1_jk"], d["beta_jk"])
f_mean, f_stat = jk(f_jk[main_data_set])                                    # (x:199), (x:199)
p_mean, p_stat = jk(f_print_jk[main_data_set])                              # (x:9), (x:9)
if len(methods) > 1:
    f_sys = excess_spread(np.array([f_jk[method] for method in methods]))   # (x:199)
    p_sys = excess_spread(np.array([f_print_jk[method] for method in methods]))     # (x:9)
else:
    f_sys = np.zeros(len(x_grid))                                           # (x:199)  one method: no spread of the methods
    p_sys = np.zeros(len(x_print))                                          # (x:9)

#---- save
file = os.path.join(save_dir, pdf_name.format(**name_args))
with h5py.File(file, "w") as f:
    f["x"] = x_grid                                                         # (x:199)
    for data_set in data_sets:
        f[f"f_jk/{data_set}"] = f_jk[data_set]                              # (jk:1504, x:199)
    f["f_mean"] = f_mean                                                    # (x:199)
    f["f_stat"] = f_stat                                                    # (x:199)
    f["f_sys"] = f_sys                                                      # (x:199)
    f["settings"] = settings_text
print("saved", file)

#---- print
print(f"\n{hadron}, {data_tag}, {main_data_set}:  f(x) = x g(x)/<x>_g,  mean (statistical error)" +
      (" (systematic);  then every method alone, mean (statistical)" if len(methods) > 1 else ""))
for i in range(len(x_print)):
    line = f"   x {x_print[i]:.2f}:  {p_mean[i]:.4f} ({p_stat[i]:.4f})"
    if len(methods) > 1:
        line = line + f" ({p_sys[i]:.4f})   "
        for method in methods:
            m_mean, m_stat = jk(f_print_jk[method][:, i])
            line = line + f"  {method} {m_mean:.4f} ({m_stat:.4f})"
    print(line)

#---- plot
f_tot = np.sqrt(f_stat ** 2 + f_sys ** 2)                                   # (x:199)
fig, ax = plt.subplots(figsize=(7.5, 5.2))
if len(methods) > 1:
    ax.fill_between(x_grid, f_mean - f_tot, f_mean + f_tot, color="#3b2f63", alpha=0.15, lw=0, label="statistical + systematic")
ax.fill_between(x_grid, f_mean - f_stat, f_mean + f_stat, color="#3b2f63", alpha=0.3, lw=0, label="statistical")
ax.plot(x_grid, f_mean, color="#3b2f63", lw=2)
if len(methods) > 1:
    for method, color in zip(methods, ["#2a78d6", "#eb6834", "#1baf7a"]):
        ax.plot(x_grid, f_jk[method].mean(axis=0), color=color, lw=1.2, ls="--", label=f"{method} alone")
ax.axvspan(0, 1 / nu_point.max(), color="#c3c2b7", alpha=0.3, lw=0, label=r"$x < 1/\nu_{\rm max}$ (set by the form of $f$)")
ax.set_xlim(0, 1)
ax.set_ylim(*pdf_ylim)
ax.set_xlabel("x")
ax.set_ylabel(r"$x\,g(x)\,/\,\langle x \rangle_g$")
ax.set_title(f"{hadron}, {data_tag}, {main_data_set}, {terms_tag}, svdcut {fit_svdcut:g}", fontsize=10)
ax.grid(color="#e1e0d9", lw=0.5)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
save_figure(fig, f"P5_gluon_pdf_{data_tag}_{main_data_set}_{terms_tag}_svd{fit_svdcut:g}")

## P6. Several saved PDFs in one plot

In [ ]:
#======== P6. several saved PDFs in one plot:  one panel per hadron, one curve per data tag;  the files of P5 (run sections 0 - P5 for each of them first) ========
compare_hadrons = ["pion", "proton"]
compare_data_tags = ["zero_flow_line_t2.0-4.0"]
compare_colors = ["#eb6834", "#1baf7a", "#2a78d6", "#eda100"]
compare_ylim = (0, 6)

fig, axes = plt.subplots(1, len(compare_hadrons), figsize=(6.5 * len(compare_hadrons), 5), sharey=True, squeeze=False)
for i_h in range(len(compare_hadrons)):
    ax = axes[0, i_h]
    name = compare_hadrons[i_h]
    for tag, color in zip(compare_data_tags, compare_colors):
        file = os.path.join(save_dir, pdf_name.format(**name_args)).replace(data_tag, tag)                  #the P5 file with the other data tag ...
        file = file.replace(f"{hadron}_forward_ncfg{ncfg}", f"{name}_forward_ncfg{ncfg}")                   #... and the other hadron
        file = file.replace(run_tag, f"forward_ncfg{ncfg}_" + {"pion": "pion", "proton": "proton_projected"}[name])
        if not os.path.exists(file):
            print("not there:", file)
            continue
        with h5py.File(file, "r") as f:
            x = f["x"][()]                                                  # (x:199)
            mean = f["f_mean"][()]                                          # (x:199)
            stat = f["f_stat"][()]                                          # (x:199)
            sys_err = f["f_sys"][()]                                        # (x:199)
        tot = np.sqrt(stat ** 2 + sys_err ** 2)                             # (x:199)
        ax.fill_between(x, mean - tot, mean + tot, color=color, alpha=0.12, lw=0)
        ax.fill_between(x, mean - stat, mean + stat, color=color, alpha=0.28, lw=0)
        ax.plot(x, mean, color=color, lw=2, label=tag)
    ax.set_xlim(0, 1)
    ax.set_ylim(*compare_ylim)
    ax.set_xlabel("x")
    ax.set_title(f"{name}, {main_data_set}:  " + ("dark band statistical, light band statistical + systematic" if len(methods) > 1 else "band: statistical error"), fontsize=10)
    ax.grid(color="#e1e0d9", lw=0.5)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=9, frameon=False)
axes[0, 0].set_ylabel(r"$x\,g(x)\,/\,\langle x \rangle_g$")
fig.tight_layout()
save_figure(fig, "P6_gluon_pdf_" + main_data_set + "_" + "_".join(compare_data_tags))